# `DPR processing` and `Auxip staging` Prefect flows

  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-797
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-798
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-799

## Initialisation

In [ ]:
# Imports
from dataclasses import asdict
import os
import os.path as osp
from pystac.asset import Asset

from resources.widget_utils import (
    dpr_proc_radio, deploy_prefect_radio, deploy_prefect, run_prefect_radio, run_prefect, shutdown_checkbox)

from rs_common.prefect_utils import *
from rs_workflows.flow_utils import  DprProcessIn, Priority, ProcessingMode, ProcessorEnum, WorkflowType
from rs_workflows.init_pi_db_flow import init_pi_database
from rs_workflows.on_demand_processing import dpr_processing, on_demand_cadip_staging

In [ ]:
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard_url = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard_url}")

In [ ]:
# Choose prefect deployment method
deploy_prefect_radio

In [ ]:
# Choose prefect flow run method
run_prefect_radio

In [ ]:
# Choose dpr processor
dpr_proc_radio

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_staging(scale=2)

# Init the processor dask cluster. 
# NOTE: use little resources for now because we only call the task tables.
print(f"** Init Dask cluster for: {dpr_proc_radio.value.name!r} **")
match dpr_proc_radio.value:
    case ProcessorEnum.MOCKUP:
        init_dask_cluster_mockup(scale=1)
    case ProcessorEnum.S1L0 | ProcessorEnum.S3L0:
        init_dask_cluster_l0(scale=1)
    case ProcessorEnum.S1ARD:
        init_dask_cluster_s1ard(scale=1)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_staging)
display(dask_cluster_eopf)

In [ ]:
# Get the prefect share bucket folder
share_bucket, _ = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")
s3_code_folder = f"users/{OWNER_ID}/code"

In [ ]:
# Create test collections
INPUT_COLLECTION = "TEST_FLOW_INPUT"
AUXIP_COLLECTION = "TEST_FLOW_AUXIP"
OUTPUT_COLLECTION = "TEST_FLOW_OUTPUT"
for collection in (INPUT_COLLECTION, AUXIP_COLLECTION, OUTPUT_COLLECTION):
  create_test_collection(collection)
  
# Prefect flow environment arguments
flow_env_args = {
  "env": {
    "owner_id": OWNER_ID,
  },
}

# DPR processing input parameters
dpr_process_in = DprProcessIn(
    **flow_env_args, 
    processor_name=dpr_proc_radio.value, 
    processor_version="", # NOTE: is it used ?
    dask_cluster_label=cluster_info_eopf.cluster_label,
    pipeline = "set_me_later",
    unit = None,
    priority = Priority.LOW,
    workflow_type = WorkflowType.ON_DEMAND,
    input_products = {},
    generated_product_to_collection_identifier = {"*": AUXIP_COLLECTION},
    auxiliary_product_to_collection_identifier = {"*": OUTPUT_COLLECTION},
    processing_mode = [ProcessingMode.ALWAYS],
    start_datetime = None, 
    end_datetime = None,
    satellite=None,
)

## Deploy and run INIT PI DB flow

In [ ]:
# Deploy the Prefect flow
pi_deploy = await deploy_prefect(
    "../../sprint27/init_pi_db_flows.yaml", s3_code_folder, os.environ["PREFECT_WORK_POOL_GENERAL"]
)

In [ ]:
# Run the Prefect flow
await run_prefect(pi_deploy, init_pi_database, flow_env_args)

## Deploy rs-client-libraries Prefect flows

In [ ]:
# Deploy the Prefect flows
dpr_processing_deploy, auxip_deploy, cadip_deploy = await deploy_prefect(
    "./dpr_processing_flow.yaml", s3_code_folder, os.environ["PREFECT_WORK_POOL_EOPF"]
)

## Init the L0 demos

In [ ]:
if dpr_proc_radio.value in (ProcessorEnum.S1L0, ProcessorEnum.S3L0):
    print(f"Init demo for: {dpr_proc_radio.value.name!r}")

    if dpr_proc_radio.value == ProcessorEnum.S1L0:
        cadip_collection = "sgs_sentinel1"
        cadip_session = "S1A_20200105072204051312"
    else:
        cadip_collection = "sgs_sentinel3"
        cadip_session = "S3B_20251010143722593812"

    # Stage a cadip session
    params = {
        **flow_env_args,
        "cadip_collection_identifier": cadip_collection,
        "session_identifier": cadip_session,
        "catalog_collection_identifier": INPUT_COLLECTION,
    }    
    await run_prefect(cadip_deploy, on_demand_cadip_staging, params)

    # Update the input product list of the dpr processing
    dpr_process_in.input_products = {cadip_session: INPUT_COLLECTION}

## Init the S1-ARD demo

In [ ]:
# We need to add some fake input data in the catalog
if dpr_proc_radio.value == ProcessorEnum.S1ARD:
    print(f"Init demo for: {dpr_proc_radio.value.name!r}")

    geometry = {
        "type": "Polygon",
        "coordinates": [[[-180, -90], [180, -90], [180, 90], [-180, 90], [-180, -90]]],
    }
    bbox = [-180.0, -90.0, 180.0, 90.0]
    now = datetime.now()
    properties = {}

    for item_id in {
        "S1A_IW_SLC__1SDV_20240416T171518_20240416T171545_053462_067C88_DA9D",
    }:        
        assets = {
            f"{item_id}.SAFE": Asset(href=f"s3://rs-dev-cluster-temp/ARD_V2/SAFE/{item_id}.SAFE")}
        item = Item(
            id=item_id, geometry=geometry, bbox=bbox, datetime=now, properties=properties, assets=assets)
        
        # print(f"Publish item: {json.dumps(item.to_dict(), indent=2)}")
        #catalog_client.add_item(INPUT_COLLECTION, item)

        # Update the input product list of the dpr processing
        #dpr_process_in.input_products[item_id] = INPUT_COLLECTION


<div class="alert alert-block alert-warning">

Note: this S1ARD init does not work. In local mode it fails with:

```
Detail: Failed to transfer file(s) from 'rs-dev-cluster-temp' bucket to 'rs-dev-cluster-catalog' catalog bucket!
```

Because the assets are store in the cluster bucket. We'll need to discuss how to work in local mode with these assets.

In cluster mode it fails with a timeout error, I don't know why. 

So for now just do the same init as L0 and we'll discuss this later.
</div>

In [ ]:
if dpr_proc_radio.value == ProcessorEnum.S1ARD:
    cadip_collection = "sgs_sentinel1"
    cadip_session = "S1A_20200105072204051312"
    params = {
        **flow_env_args,
        "cadip_collection_identifier": cadip_collection,
        "session_identifier": cadip_session,
        "catalog_collection_identifier": INPUT_COLLECTION,
    }    
    await run_prefect(cadip_deploy, on_demand_cadip_staging, params)
    dpr_process_in.input_products = {cadip_session: INPUT_COLLECTION}

## Run the DPR processing flow

In [ ]:
# TO BE DISCUSSED: what should we test ?
# For now put the full pipeline depending on the processor.
match dpr_proc_radio.value:
    case ProcessorEnum.MOCKUP:
        dpr_process_in.pipeline = ""
    case ProcessorEnum.S1L0:
        dpr_process_in.pipeline = "s1_l0_full"
    case ProcessorEnum.S3L0:
        dpr_process_in.pipeline = "s3_l0_full"
    case ProcessorEnum.S1ARD:
        dpr_process_in.pipeline = "s1_ard_full"

In [ ]:
# Be sure to use the processor chosen by the user
dpr_process_in.processor_name=dpr_proc_radio.value
print(f"Run demo for: {dpr_proc_radio.value.name!r}")

# Run the processor
params = {"dpr_input": asdict(dpr_process_in)}
await run_prefect(dpr_processing_deploy, dpr_processing, params)

In [ ]:
# Get the processing unit list from the last flow run artifacts
# See: https://docs-3.prefect.io/v3/api-ref/rest-api/server/artifacts/read-latest-artifact
response = http_session.get(f"{os.environ['PREFECT_API_URL']}/artifacts/units-list/latest")
response.raise_for_status()
contents = response.json()

# Render the artifact as markdown.
# TO BE DISCUSSED: do we want to save the artifact as markdown 
# or pure json that could be used easier from python code ?
from IPython.display import Markdown
display(Markdown(contents["data"]))


<div class="alert alert-block alert-warning">

TO BE DISCUSSED: in dpr_processing could we save the cql queries as artifacts ? 

Then retrieve them from this demo to run the auxip staging separately from the dpr_processing ?
</div>

## Shutdown the dask clusters

In [ ]:
# Choose to shutdown the dask cluster
shutdown_checkbox

In [ ]:
if shutdown_checkbox.value:
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)
    close_dask_clusters()
# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.